# Machine Learning for Breast Cancer Diagnosis from DNA Methylation Data

**Bachelor's thesis – Mahdad Naderinia – University of Messina (Supervisor: Prof. Daniele Ravi)**

This notebook contains the complete code of the project:

1. Setup
2. Download and preprocessing of the TCGA BRCA methylation data
3. Exploratory data analysis (EDA)
4. Train–test split (grouped by patient)
5. Model comparison with patient-grouped cross-validation
6. Evaluation on the test set (confusion matrices, ROC curves)
7. Computational time comparison
8. Hyperparameter tuning and final model
9. Number of CpG sites needed
10. Most important CpG sites
11. Learning curve, effect of regularization and stability
12. Saving the final model

Run the cells from top to bottom (*Runtime → Run all*). The full notebook takes about 10–20 minutes on Google Colab.

## 1. Setup

In [ ]:
# XGBoost and seaborn are pre-installed on Google Colab. Uncomment the next line if you run elsewhere.
# !pip install -q xgboost seaborn

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.model_selection import (StratifiedGroupKFold, cross_validate, GridSearchCV,
                                     learning_curve)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (make_scorer, recall_score, precision_score, f1_score, accuracy_score,
                             balanced_accuracy_score, roc_auc_score, roc_curve, confusion_matrix,
                             classification_report)
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings("ignore")

# Colors used in all figures: blue = healthy, orange = cancer
COLORS = {0: "#2a78d6", 1: "#eb6834"}
NAMES = {0: "Healthy", 1: "Cancer"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})

RAW_FILE = "brca_methylation450.gz"
DATA_FILE = "brca_methylation_final.csv"
os.makedirs("figures", exist_ok=True)

## 2. Download and preprocessing of the TCGA data

The raw data come from the UCSC Xena platform (dataset `TCGA.BRCA.sampleMap/HumanMethylation450`, 485,578 CpG probes × 888 samples).

Preprocessing steps:
1. remove probes with missing values;
2. keep the 5,000 most variable probes (unsupervised – no labels used);
3. remove the 63 SNP control probes (`rs...`), which measure genetic variants and not methylation;
4. keep only primary tumors (barcode code `01`) and solid tissue normal samples (code `11`);
5. create the label (`Cancer_Status`: 1 = cancer, 0 = healthy) and the patient identifier.

If `brca_methylation_final.csv` already exists (for example uploaded to Colab), this step is skipped.

In [ ]:
if not os.path.exists(DATA_FILE):
    if not os.path.exists(RAW_FILE):
        url = ("https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/"
               "TCGA.BRCA.sampleMap%2FHumanMethylation450.gz")
        os.system(f'wget -q -O {RAW_FILE} "{url}"')

    chunks = []
    for chunk in pd.read_csv(RAW_FILE, sep="\t", index_col=0, chunksize=20000, compression="gzip"):
        chunks.append(chunk.dropna().astype("float32"))          # 1. drop probes with missing values
    meth = pd.concat(chunks)
    print("Probes without missing values:", meth.shape)

    top = meth.var(axis=1).nlargest(5000).index                  # 2. 5,000 most variable probes
    X_all = meth.loc[top].T                                       # rows = samples, columns = probes
    X_all = X_all.loc[:, ~X_all.columns.str.startswith("rs")]     # 3. remove SNP control probes

    X_all = X_all[X_all.index.str[-2:].isin(["01", "11"])]        # 4. tumor (01) and normal (11)
    labels = (X_all.index.str[-2:] == "01").astype(int)          # 5. 1 = cancer, 0 = healthy
    patients = X_all.index.str[:12]

    X_all.assign(Cancer_Status=labels, Patient_ID=patients).to_csv(DATA_FILE, index_label="Sample_ID")
    print("Saved", DATA_FILE)
else:
    print(DATA_FILE, "already exists – download and preprocessing skipped.")

## 3. Exploratory data analysis
### 3.1 Load the final dataset and quality checks

In [ ]:
df = pd.read_csv(DATA_FILE, index_col="Sample_ID")
y = df["Cancer_Status"]
groups = df["Patient_ID"]
X = df.drop(columns=["Cancer_Status", "Patient_ID"]).astype("float32")

print("Shape (samples, CpG sites):", X.shape)
print("Missing values:", int(X.isna().sum().sum()))
print("Duplicated samples:", int(df.index.duplicated().sum()))
print(f"Value range: {X.values.min():.3f} – {X.values.max():.3f}")
print("Class counts:", y.value_counts().to_dict())
X.iloc[:5, :5]

### 3.2 Class distribution

In [ ]:
counts = y.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar([NAMES[i] for i in counts.index], counts.values,
              color=[COLORS[i] for i in counts.index], width=0.55, zorder=3)
ax.bar_label(bars, labels=[f"{v} ({v / counts.sum():.1%})" for v in counts.values], padding=4)
ax.set_ylabel("Number of samples"); ax.set_xlabel("Class")
ax.set_title(f"Class distribution of the dataset (n = {counts.sum()})")
ax.set_ylim(0, counts.max() * 1.15); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.savefig("figures/class_balance.png", dpi=300); plt.show()

### 3.3 Patients and paired samples

In [ ]:
per_patient = df.groupby("Patient_ID")["Cancer_Status"].agg(["count", "nunique"])
print("Unique patients:", len(per_patient))
print("Patients with both a tumor and a normal sample:", int((per_patient["nunique"] == 2).sum()))

### 3.4 Distribution of beta values

In [ ]:
bins = np.linspace(0, 1, 51)
fig, ax = plt.subplots(figsize=(8, 4.5))
for c in [1, 0]:
    vals = X[y == c].values.ravel()
    ax.hist(vals, bins=bins, density=True, alpha=0.45, color=COLORS[c], zorder=2)
    ax.hist(vals, bins=bins, density=True, histtype="step", lw=2, color=COLORS[c],
            label=f"{NAMES[c]} (n = {(y == c).sum()})", zorder=3)
ax.set_xlabel("Beta value (0 = unmethylated, 1 = fully methylated)"); ax.set_ylabel("Density")
ax.set_title("Distribution of methylation (beta) values"); ax.set_xlim(0, 1)
handles, lbls = ax.get_legend_handles_labels(); ax.legend(handles[::-1], lbls[::-1], frameon=False)
plt.tight_layout(); plt.savefig("figures/beta_distribution.png", dpi=300); plt.show()

### 3.5 Feature statistics and differential methylation

In [ ]:
stats = pd.DataFrame({"mean": X.mean(), "std": X.std()})
print(stats.describe().round(3))
print("Low (<0.3):", (stats["mean"] < 0.3).sum(),
      "| intermediate (0.3–0.7):", stats["mean"].between(0.3, 0.7).sum(),
      "| high (>0.7):", (stats["mean"] > 0.7).sum())

diff = X[y == 1].mean() - X[y == 0].mean()          # mean beta in cancer minus mean beta in healthy
print("\nSites with |difference| > 0.2:", (diff.abs() > 0.2).sum())
print("Hypermethylated in cancer:", (diff > 0.2).sum(), "| Hypomethylated:", (diff < -0.2).sum())

top6 = list(diff.nlargest(3).index) + list(diff.nsmallest(3).index)
fig, axes = plt.subplots(1, 6, figsize=(15, 3.8), sharey=True)
for ax, site in zip(axes, top6):
    bp = ax.boxplot([X.loc[y == 0, site], X.loc[y == 1, site]], widths=0.55, patch_artist=True,
                    showfliers=False, medianprops=dict(color="black"))
    for patch, c in zip(bp["boxes"], [0, 1]):
        patch.set_facecolor(COLORS[c]); patch.set_alpha(0.75)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["Healthy", "Cancer"])
    ax.set_title(f"{site}\n{'gained' if diff[site] > 0 else 'lost'} in cancer", fontsize=10)
axes[0].set_ylabel("Beta value")
plt.tight_layout(); plt.savefig("figures/top_sites_boxplots.png", dpi=300); plt.show()

### 3.6 Heatmap of the 50 most different CpG sites

In [ ]:
top50 = diff.abs().nlargest(50).index
hyper = diff[top50][diff[top50] > 0].sort_values(ascending=False).index
hypo = diff[top50][diff[top50] < 0].sort_values().index
sites = list(hyper) + list(hypo)
score = X[hyper].mean(axis=1)
order = list(score[y == 0].sort_values().index) + list(score[y == 1].sort_values().index)
n_h, n_c = (y == 0).sum(), (y == 1).sum()

fig = plt.figure(figsize=(11, 6.5))
gs = fig.add_gridspec(2, 2, height_ratios=[0.6, 12], width_ratios=[40, 1], hspace=0.04, wspace=0.03)
ax_bar, ax, cax = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
ax_bar.imshow(y.loc[order].values[None, :], aspect="auto", interpolation="nearest",
              cmap=ListedColormap([COLORS[0], COLORS[1]]))
ax_bar.set_xticks([]); ax_bar.set_yticks([]); ax_bar.grid(False)
ax_bar.text(n_h / 2, -0.9, f"Healthy (n={n_h})", ha="center", va="bottom")
ax_bar.text(n_h + n_c / 2, -0.9, f"Cancer (n={n_c})", ha="center", va="bottom")
im = ax.imshow(X.loc[order, sites].T.values, aspect="auto", cmap="Blues", vmin=0, vmax=1,
               interpolation="nearest")
ax.axvline(n_h - 0.5, color="black", lw=1.5); ax.axhline(len(hyper) - 0.5, color="white", lw=2)
ax.set_xticks([]); ax.grid(False)
ax.set_yticks([(len(hyper) - 1) / 2, len(hyper) + (len(hypo) - 1) / 2])
ax.set_yticklabels([f"Gained in cancer\n({len(hyper)} sites)", f"Lost in cancer\n({len(hypo)} sites)"])
ax.set_xlabel("Samples")
fig.colorbar(im, cax=cax).set_label("Beta value")
fig.suptitle("Methylation of the 50 most different CpG sites", y=0.97)
plt.savefig("figures/heatmap_50_sites.png", dpi=300, bbox_inches="tight"); plt.show()

### 3.7 Principal Component Analysis

In [ ]:
pca = PCA(n_components=2)
Z = pca.fit_transform(StandardScaler().fit_transform(X))
fig, ax = plt.subplots(figsize=(7, 5.5))
for c in [1, 0]:
    ax.scatter(Z[y == c, 0], Z[y == c, 1], s=22, color=COLORS[c], alpha=0.7, edgecolors="white",
               linewidths=0.6, label=f"{NAMES[c]} (n={(y == c).sum()})")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} of variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} of variance)")
ax.set_title("PCA of DNA methylation profiles"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("figures/pca.png", dpi=300); plt.show()
print(f"PC1 + PC2 explain {pca.explained_variance_ratio_.sum():.1%} of the variance")

## 4. Train–test split

20% of the **patients** are set aside as an independent test set. `StratifiedGroupKFold` keeps the class proportions and makes sure that no patient appears in both sets. The test set is used only once, at the end.

In [ ]:
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(outer.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]

print("Training set:", X_train.shape, y_train.value_counts().to_dict())
print("Test set:    ", X_test.shape, y_test.value_counts().to_dict())
print("Patients in both sets:", len(set(g_train) & set(groups.iloc[test_idx])))   # must be 0

## 5. Model comparison with patient-grouped cross-validation

Seven classifiers are compared with 5-fold `StratifiedGroupKFold` on the training set. Scaling is placed inside a `Pipeline`, so it is fitted only on the training folds (no data leakage). Class imbalance is handled with balanced class weights.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
scoring = {"ROC-AUC": "roc_auc", "Accuracy": "accuracy", "Balanced accuracy": "balanced_accuracy",
           "Sensitivity": make_scorer(recall_score, pos_label=1),
           "Specificity": make_scorer(recall_score, pos_label=0), "F1-score": "f1"}
spw = (y_train == 0).sum() / (y_train == 1).sum()          # weight of the cancer class for XGBoost

def make_models():
    return {
        "Dummy (baseline)": DummyClassifier(strategy="stratified", random_state=42),
        "Logistic Regression": make_pipeline(StandardScaler(),
            LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)),
        "SVM (linear)": make_pipeline(StandardScaler(),
            SVC(kernel="linear", C=0.01, class_weight="balanced", probability=True, random_state=42)),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.1,
                                 scale_pos_weight=spw, eval_metric="logloss", random_state=42, n_jobs=-1),
        "MLP": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,), alpha=1.0,
                                                             max_iter=1000, random_state=42)),
    }

cv_rows, cv_raw = [], {}
for name, model in make_models().items():
    s = cross_validate(model, X_train, y_train, groups=g_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_raw[name] = s
    cv_rows.append({"Model": name, **{m: f"{s['test_' + m].mean():.3f} ± {s['test_' + m].std():.3f}"
                                      for m in scoring}})
    print(f"{name:20s} done")
cv_table = pd.DataFrame(cv_rows).set_index("Model")
cv_table

In [ ]:
# Figure: sensitivity, specificity and balanced accuracy of each model (without the baseline)
names = [n for n in cv_raw if n != "Dummy (baseline)"]
names = sorted(names, key=lambda n: cv_raw[n]["test_Balanced accuracy"].mean())
yy = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 5.4))
for metric, col, off, marker in [("Sensitivity", COLORS[1], 0.15, "o"), ("Specificity", COLORS[0], -0.15, "o"),
                                 ("Balanced accuracy", "#222222", 0, "D")]:
    m = np.array([cv_raw[n]["test_" + metric].mean() for n in names])
    s = np.array([cv_raw[n]["test_" + metric].std() for n in names])
    ax.errorbar(m, yy + off, xerr=[np.minimum(s, m), np.minimum(s, 1 - m)], fmt=marker, color=col,
                capsize=3, ms=7, label=metric)
ax.set_yticks(yy); ax.set_yticklabels(names); ax.set_xlim(0.8, 1.005)
ax.set_xlabel("Score (mean ± std over 5 patient-grouped CV folds)")
ax.set_title("Model comparison on the training set")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.45, -0.16), ncol=3)
plt.tight_layout(); plt.savefig("figures/model_comparison.png", dpi=300); plt.show()

## 6. Evaluation on the test set
Each model is retrained on the whole training set and evaluated once on the test set. The training and prediction times are also measured (Section 7).

In [ ]:
fitted, probs, preds, times, test_rows = {}, {}, {}, {}, []
for name, model in make_models().items():
    t0 = time.perf_counter(); model.fit(X_train, y_train); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); p = model.predict_proba(X_test)[:, 1]; t_pred = time.perf_counter() - t0
    pr = model.predict(X_test)
    fitted[name], probs[name], preds[name], times[name] = model, p, pr, (t_fit, t_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, pr).ravel()
    test_rows.append({"Model": name, "ROC-AUC": roc_auc_score(y_test, p), "Accuracy": accuracy_score(y_test, pr),
                      "Balanced acc.": balanced_accuracy_score(y_test, pr), "Sensitivity": recall_score(y_test, pr),
                      "Specificity": recall_score(y_test, pr, pos_label=0), "Precision": precision_score(y_test, pr),
                      "F1": f1_score(y_test, pr), "FP": fp, "FN": fn})
test_table = pd.DataFrame(test_rows).set_index("Model").round(3)
test_table

In [ ]:
# Confusion matrices of the six models
model_names = [n for n in fitted if n != "Dummy (baseline)"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7.6))
for ax, n in zip(axes.ravel(), model_names):
    cm = confusion_matrix(y_test, preds[n])
    ax.imshow(cm, cmap="Blues"); ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16, fontweight="bold",
                    color="white" if cm[i, j] > 80 else "#222222")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Healthy", "Cancer"]); ax.set_xlabel("Predicted")
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Healthy", "Cancer"]); ax.set_ylabel("Actual")
    ax.set_title(f"{n}\nbalanced acc. = {balanced_accuracy_score(y_test, preds[n]):.3f}")
fig.suptitle(f"Confusion matrices on the test set ({(y_test == 1).sum()} cancer, {(y_test == 0).sum()} healthy)",
             fontweight="bold")
plt.tight_layout(); plt.savefig("figures/confusion_matrices.png", dpi=300); plt.show()

In [ ]:
# ROC curves (full view and zoom on the top-left corner)
line_colors = {"Logistic Regression": "#2a78d6", "SVM (linear)": "#eb6834", "KNN": "#1baf7a",
               "Random Forest": "#eda100", "XGBoost": "#e87ba4", "MLP": "#6250d6", "Dummy (baseline)": "#888888"}
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, zoom in zip(axes, [False, True]):
    for n in fitted:
        fpr, tpr, _ = roc_curve(y_test, probs[n])
        ax.plot(fpr, tpr, color=line_colors[n], lw=2, ls="--" if n.startswith("Dummy") else "-",
                label=f"{n} (AUC = {roc_auc_score(y_test, probs[n]):.3f})")
    if zoom:
        ax.set_xlim(-0.005, 0.3); ax.set_ylim(0.85, 1.005); ax.set_title("Zoom on the top-left corner")
    else:
        ax.plot([0, 1], [0, 1], color="#cccccc"); ax.set_title("Full ROC curves")
        ax.legend(frameon=False, fontsize=9, loc="lower right")
    ax.set_xlabel("False positive rate (1 − specificity)"); ax.set_ylabel("True positive rate (sensitivity)")
fig.suptitle("ROC curves on the test set", fontweight="bold")
plt.tight_layout(); plt.savefig("figures/roc_curves.png", dpi=300); plt.show()

## 7. Computational time comparison

In [ ]:
time_table = pd.DataFrame({n: {"Training time (s)": t[0], "Prediction time (s)": t[1]}
                           for n, t in times.items() if not n.startswith("Dummy")}).T.sort_values("Training time (s)")
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(time_table.index, time_table["Training time (s)"], color=COLORS[0], height=0.55, zorder=3)
ax.bar_label(bars, labels=[f"{v:.2f} s" for v in time_table["Training time (s)"]], padding=4)
ax.set_xlabel("Training time on the training set (seconds)"); ax.grid(axis="y", visible=False)
ax.set_title(f"Training time of each model ({X_train.shape[0]} samples × {X_train.shape[1]:,} features)")
plt.tight_layout(); plt.savefig("figures/training_time.png", dpi=300); plt.show()
time_table.round(3)

## 8. Hyperparameter tuning and final model
Logistic Regression is selected as the final model. Its regularization parameter C is tuned with grid search inside patient-grouped cross-validation, then the model is evaluated once on the test set.

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
grid = GridSearchCV(pipe, {"clf__C": [0.001, 0.01, 0.1, 1, 10]}, scoring="roc_auc", cv=cv, n_jobs=1)
grid.fit(X_train, y_train, groups=g_train)
print(pd.DataFrame({"C": grid.cv_results_["param_clf__C"], "mean ROC-AUC": grid.cv_results_["mean_test_score"],
                    "std": grid.cv_results_["std_test_score"]}).round(4).to_string(index=False))
print("\nBest C:", grid.best_params_["clf__C"])
final_model = grid.best_estimator_

proba = final_model.predict_proba(X_test)[:, 1]
pred = final_model.predict(X_test)
print(f"\nTest ROC-AUC: {roc_auc_score(y_test, proba):.3f} | accuracy: {accuracy_score(y_test, pred):.3f} | "
      f"balanced accuracy: {balanced_accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, target_names=["Healthy", "Cancer"], digits=3))

# Misclassified test samples
wrong = X_test.index[pred != y_test.values]
print("Misclassified samples:")
for s in wrong:
    print(f"  {s}: true = {NAMES[int(y_test[s])]}, predicted probability of cancer = {proba[list(X_test.index).index(s)]:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
cm = confusion_matrix(y_test, pred); ax = axes[0]
ax.imshow(cm, cmap="Blues"); ax.grid(False)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=18, fontweight="bold",
                color="white" if cm[i, j] > 80 else "#222222")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Healthy", "Cancer"]); ax.set_xlabel("Predicted")
ax.set_yticks([0, 1]); ax.set_yticklabels(["Healthy", "Cancer"]); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix")
fpr, tpr, _ = roc_curve(y_test, proba); ax = axes[1]
ax.plot(fpr, tpr, color=COLORS[0], lw=2.5, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, proba):.3f})")
ax.plot([0, 1], [0, 1], "--", color="#aaaaaa", label="Random classifier (AUC = 0.5)")
ax.set_xlabel("False positive rate (1 − specificity)"); ax.set_ylabel("True positive rate (sensitivity)")
ax.set_title("ROC curve"); ax.legend(frameon=False, loc="lower right")
fig.suptitle(f"Final model on the test set (Logistic Regression, C = {grid.best_params_['clf__C']})", fontweight="bold")
plt.tight_layout(); plt.savefig("figures/final_model.png", dpi=300); plt.show()

## 9. How many CpG sites are needed?
Logistic Regression is trained with only the *k* most informative sites (ANOVA F-test). The selection is **inside the pipeline**, so it is repeated in every fold using only training data.

In [ ]:
ks = [1, 2, 5, 10, 20, 50, 100, 500, X.shape[1]]
k_rows = []
for k in ks:
    pk = Pipeline([("select", SelectKBest(f_classif, k=k)), ("scaler", StandardScaler()),
                   ("clf", LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))])
    s = cross_validate(pk, X_train, y_train, groups=g_train, cv=cv,
                       scoring={"auc": "roc_auc", "bal": "balanced_accuracy"}, n_jobs=1)
    pk.fit(X_train, y_train)
    k_rows.append({"k": k, "ROC-AUC (CV)": s["test_auc"].mean(), "std AUC": s["test_auc"].std(),
                   "Bal. acc. (CV)": s["test_bal"].mean(), "std bal": s["test_bal"].std(),
                   "ROC-AUC (test)": roc_auc_score(y_test, pk.predict_proba(X_test)[:, 1]),
                   "Bal. acc. (test)": balanced_accuracy_score(y_test, pk.predict(X_test))})
    print(f"{k:>5} sites: CV ROC-AUC = {k_rows[-1]['ROC-AUC (CV)']:.4f}")
k_table = pd.DataFrame(k_rows)

fig, ax = plt.subplots(figsize=(8, 4.6))
for col, sd, c, mk, lab in [("ROC-AUC (CV)", "std AUC", COLORS[0], "o", "ROC-AUC"),
                            ("Bal. acc. (CV)", "std bal", COLORS[1], "D", "Balanced accuracy")]:
    m, s = k_table[col].values, k_table[sd].values
    ax.errorbar(k_table["k"], m, yerr=[np.minimum(s, m), np.minimum(s, 1 - m)], marker=mk, color=c,
                lw=2, capsize=3, label=lab)
ax.set_xscale("log"); ax.set_xticks(ks); ax.set_xticklabels(ks)
ax.set_xlabel("Number of CpG sites used (selected inside cross-validation)"); ax.set_ylabel("Score (5-fold CV)")
ax.set_title("Performance vs. number of CpG sites"); ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig("figures/number_of_sites.png", dpi=300); plt.show()
k_table.round(3)

## 10. Most important CpG sites

In [ ]:
F, pvals = f_classif(X_train, y_train)
ranking = pd.DataFrame({"F-statistic": F}, index=X.columns).sort_values("F-statistic", ascending=False).head(20)
ranking["Mean beta (healthy)"] = X_train[y_train == 0][ranking.index].mean()
ranking["Mean beta (cancer)"] = X_train[y_train == 1][ranking.index].mean()
ranking["Change in cancer"] = np.where(ranking["Mean beta (cancer)"] > ranking["Mean beta (healthy)"],
                                       "Hypermethylated", "Hypomethylated")

r = ranking.head(15).iloc[::-1]; yy = np.arange(len(r))
fig, ax = plt.subplots(figsize=(8, 6.2))
ax.hlines(yy, r["Mean beta (healthy)"], r["Mean beta (cancer)"], color="#bbbbbb", lw=2.5)
ax.scatter(r["Mean beta (healthy)"], yy, color=COLORS[0], s=70, zorder=3, label="Healthy (mean)")
ax.scatter(r["Mean beta (cancer)"], yy, color=COLORS[1], s=70, zorder=3, label="Cancer (mean)")
ax.set_yticks(yy); ax.set_yticklabels([f"{s} (F = {f:.0f})" for s, f in zip(r.index, r["F-statistic"])], fontsize=9.5)
ax.set_xlim(0, 1); ax.set_xlabel("Mean beta value (training set)")
ax.set_title("15 most discriminative CpG sites (ANOVA F-test)"); ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig("figures/top_sites.png", dpi=300); plt.show()
ranking.round(3)

In [ ]:
# Optional: map the top sites to genes with the Illumina 450K annotation from UCSC Xena
try:
    if not os.path.exists("probemap.tsv"):
        os.system('wget -q -O probemap.tsv "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/'
                  'probeMap%2FilluminaMethyl450_hg19_GPL16304_TCGAlegacy"')
    probemap = pd.read_csv("probemap.tsv", sep="\t", index_col=0)
    display(probemap.reindex(ranking.index[:10]))
except Exception as e:
    print("Gene annotation not available:", e)

## 11. Learning curve, effect of regularization and stability

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))

# Learning curve
sizes, tr_sc, va_sc = learning_curve(lr, X_train, y_train, groups=g_train, cv=cv, scoring="roc_auc",
                                     train_sizes=[0.1, 0.2, 0.35, 0.5, 0.75, 1.0])
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(sizes, tr_sc.mean(1), "o-", color=COLORS[1], lw=2, label="Training score")
ax.errorbar(sizes, va_sc.mean(1), yerr=np.minimum(va_sc.std(1), 1 - va_sc.mean(1)), fmt="o-", color=COLORS[0],
            lw=2, capsize=3, label="Validation score (CV)")
ax.set_xlabel("Number of training samples"); ax.set_ylabel("ROC-AUC")
ax.set_title("Learning curve of Logistic Regression"); ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig("figures/learning_curve.png", dpi=300); plt.show()
print(pd.DataFrame({"samples": sizes, "train": tr_sc.mean(1), "validation": va_sc.mean(1)}).round(4).to_string(index=False))

In [ ]:
# Effect of the regularization parameter C (validation curve)
Cs = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]; vc = []
for C_value in Cs:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C_value, class_weight="balanced", max_iter=5000))
    s = cross_validate(m, X_train, y_train, groups=g_train, cv=cv, scoring="roc_auc", return_train_score=True)
    vc.append([s["train_score"].mean(), s["test_score"].mean(), s["test_score"].std()])
vc = np.array(vc)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(Cs, vc[:, 0], "o-", color=COLORS[1], lw=2, label="Training score")
ax.errorbar(Cs, vc[:, 1], yerr=np.minimum(vc[:, 2], 1 - vc[:, 1]), fmt="o-", color=COLORS[0], lw=2, capsize=3,
            label="Validation score (CV)")
ax.set_xscale("log"); ax.set_xlabel("Regularization parameter C (smaller = stronger regularization)")
ax.set_ylabel("ROC-AUC"); ax.set_title("Effect of regularization on Logistic Regression")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig("figures/validation_curve.png", dpi=300); plt.show()
print(pd.DataFrame(vc, index=Cs, columns=["train", "validation", "std"]).round(4))

In [ ]:
# Stability: repeat the cross-validation with 10 different fold assignments
rep = []
for seed in range(10):
    c = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    s = cross_validate(lr, X_train, y_train, groups=g_train, cv=c, scoring={"auc": "roc_auc", "bal": "balanced_accuracy"})
    rep.append([s["test_auc"].mean(), s["test_bal"].mean()])
rep = np.array(rep)
print(f"ROC-AUC over 10 repetitions: {rep[:, 0].mean():.4f} ± {rep[:, 0].std():.4f} "
      f"(range {rep[:, 0].min():.4f}–{rep[:, 0].max():.4f})")
print(f"Balanced accuracy:           {rep[:, 1].mean():.3f} ± {rep[:, 1].std():.3f} "
      f"(range {rep[:, 1].min():.3f}–{rep[:, 1].max():.3f})")

## 12. Save the final model and the results

In [ ]:
joblib.dump(final_model, "final_logistic_regression.joblib")
cv_table.to_csv("results_cross_validation.csv")
test_table.to_csv("results_test_set.csv")
k_table.to_csv("results_number_of_sites.csv", index=False)
ranking.to_csv("top_20_cpg_sites.csv")
print("Saved: final_logistic_regression.joblib, results_*.csv, top_20_cpg_sites.csv and the figures/ folder")

# Example: load the model again and predict a new sample
model = joblib.load("final_logistic_regression.joblib")
print("Probability of cancer for the first test sample:", round(model.predict_proba(X_test.iloc[[0]])[0, 1], 3))